In [ ]:
!pip install folium
!pip install osmnx

In [ ]:
import requests

# API call to OSRM to fetch road geometries
def fetch_road_geometries(routes_list):
    """
    Fetches the road geometries for a list of routes using OSRM.
    Returns a list of GeoJSON objects.
    """
    route_data_list = []
    headers = {'User-Agent': 'VRP_Balkans_Project/1.0'}
    
    for i, route in enumerate(routes_list):
        # Format for OSRM: "lon,lat;lon,lat"
        osrm_coords = ";".join([f"{lon},{lat}" for lat, lon in route])
        
        url = f"http://router.project-osrm.org/route/v1/driving/{osrm_coords}"
        params = {"overview": "full", "geometries": "geojson"}
        
        response = requests.get(url, params=params, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            # Extract the geographic shape of the road and the distance
            geometry = data['routes'][0]['geometry']
            distance = data['routes'][0]['distance'] / 1000 # convert m to Km
            route_data_list.append({
                "route_points": route,
                "geometry": geometry,
                "distance": distance
            })
        else:
            print(f"Error fetching Route {i+1}. Status code: {response.status_code}")
            
    return route_data_list

In [ ]:
import folium

# Plotting the background map and the pre-fetched road geometries
def plot_prefetched_roads(base_map, prefetched_data):
    """
    Takes the pre-fetched road geometries and draws them on the Folium map.
    """
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'black']
    
    for i, data in enumerate(prefetched_data):
        color = colors[i % len(colors)]
        
        # 1. Draw the pre-fetched road path
        folium.GeoJson(
            data["geometry"],
            style_function=lambda feature, c=color: {
                'color': c,
                'weight': 4,
                'opacity': 0.8
            }
        ).add_to(base_map)
        
        # 2. Draw the node stops
        for stop in data["route_points"]:
            folium.CircleMarker(
                location=stop,
                radius=5,
                color=color,
                fill=True,
                fill_color='white',
                fill_opacity=1
            ).add_to(base_map)
            
    return base_map

In [ ]:
import math

# My old haversine function to calculate the geographic distance between two points on the Earth
EarthRad = 6371  # Radius of the Earth in kilometers
def haversine(city1, city2):
    lat1, lon1 = city1
    lat2, lon2 = city2
    lat1 = (lat1/180)*math.pi
    lat2 = (lat2/180)*math.pi
    lon1 = (lon1/180)*math.pi
    lon2 = (lon2/180)*math.pi
    dlat = lat1-lat2
    dlon = lon1-lon2
    di = math.sin(dlat/2)**2+math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    dfin = 2 * math.atan2(math.sqrt(di),math.sqrt(1-di))
    return dfin * EarthRad

In [ ]:
# Calculating the total distance of a route with haversine
def calculate_route_distance(route):
    distance = 0
    for i in range(len(route)-1):
        city1 = route[i]
        city2 = route[i+1]
        distance += haversine(city1, city2)
    return distance

# -> Insert the GA here

In [ ]:
# 1. Your GA finishes running and outputs the best lists of coordinates
ga_output_routes = [
    [[44.4268, 26.1025], [43.8563, 25.9560], [42.6977, 23.3219]], # Route 1
    [[42.6977, 23.3219], [43.3209, 21.8958], [44.7866, 20.4489]]  # Route 2
]

# 2. Fetch the road geometries for ONLY the final winner (Very Fast)
prefetched_roads = fetch_road_geometries(ga_output_routes)

# 3. Calculate the total distance of all routes
for route_data in prefetched_roads:
    total_distance = sum(data["distance"] for data in prefetched_roads)
print(f"distance: {total_distance}")

# 4. Create the empty map environment
balkans_map = folium.Map(location=[43.5, 22.5], zoom_start=6, tiles="OpenStreetMap")

# 5. Instantly draw the prefetched data
balkans_map = plot_prefetched_roads(balkans_map, prefetched_roads)

# Display in browser
balkans_map.save("balkans_vrp_map.html")

distance: 783.1717
